In [ ]:
# Common Python file that imports all the necessary libraries and common variables used

from setup import *

# Common link directory to save figures
out_path = "output/p3"
os.makedirs(out_path, exist_ok=True)

## Reversing the Dataset

In [ ]:
# Reversing the pairs
reversed_ds = [(fra, eng) for eng, fra in dataset]

# Rebuilding vocab and mapping for the new dataset
unique_chars = sorted(list(set(''.join([word for pair in reversed_ds for word in pair]))))
chars_to_idx = {'SOS': SOS_token, 'EOS': EOS_token}

for i, char in enumerate(unique_chars):
    chars_to_idx[char] = i + 2

idx_to_chars = {i: char for char, i in chars_to_idx.items()}

# Rebuilding dataset and loaders
reversed_torch_ds = TranslationDataset(reversed_ds, chars_to_idx)

train_size3 = int(0.8 * len(reversed_ds))
val_size3 = len(reversed_ds) - train_size3

train_reversed_ds, val_reversed_ds = random_split(reversed_torch_ds, [train_size3, val_size3], generator=torch.Generator().manual_seed(SEED))

train_reversed_loader = DataLoader(train_reversed_ds, batch_size=1, shuffle=True)
val_reversed_loader = DataLoader(val_reversed_ds, batch_size=1, shuffle=False)

## Common Encoder Class

In [ ]:
class Encoder(nn.Module):
    def __init__(self, inp_size, hidden_size):
        super(Encoder, self).__init__()

        self.input_size = inp_size
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(self.input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input_tensor, hidden_state):
        embedded = self.embedding(input_tensor).view(1, 1, -1)
        output, hidden_state = self.gru(embedded, hidden_state)

        return output, hidden_state
    
    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

## Baseline Decoder Class

In [ ]:
class Decoder(nn.Module):
    def __init__(self, out_size, hidden_size):
        super(Decoder, self).__init__()

        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(out_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, out_size)
        self.softmax = nn.LogSoftmax(dim=1)
    
    def forward(self, input_tensor, hidden_state):
        embedded = self.embedding(input_tensor).view(1, 1, -1)
        output, hidden_state = self.gru(embedded, hidden_state)
        output = self.softmax(self.out(output[0]))

        return output, hidden_state

## Attention Decoder Class

In [ ]:
class AttnDecoder(nn.Module):
    def __init__(self, hidden_size, out_size, max_length=100, dropout_p=0.1):
        super(AttnDecoder, self).__init__()

        self.hidden_size = hidden_size
        self.output_size = out_size
        self.drouput_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(out_size, hidden_size)

        # Added parts for attention
        self.attn = nn.Linear(self.hidden_size * 2, self.max_length)
        self.attn_combine = nn.Linear(self.hidden_size * 2, self.hidden_size)
        self.drouput = nn.Dropout()


        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, out_size)
    
    def forward(self, input_tensor, hidden_state, encoder_outputs):
        embedded = self.embedding(input_tensor).view(1, 1, -1)
        embedded = self.drouput(embedded)

        # Calculating weights with attentionn
        attn_weights = torch.softmax(self.attn(torch.cat((embedded[0], hidden_state[0]), 1)), dim=1)

        # Getting context from encoder outputs
        attn_applied = torch.bmm(attn_weights.unsqueeze(0), encoder_outputs.unsqueeze(0))

        output = torch.cat((embedded[0], attn_applied[0]), 1)
        output = self.attn_combine(output).unsqueeze(0)
        output = torch.relu(output)
        output, hidden_state = self.gru(output, hidden_state)
        output = torch.log_softmax(self.out(output[0]), dim=1)

        return output, hidden_state, attn_weights
    
    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

## Baseline Helper Functions

In [ ]:
def forward_pass(input_tensor, target_tensor, encoder, decoder, criterion):
    encoder_hidden = encoder.initHidden()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)
    loss = 0

    # Encoding
    # Encoding loop
    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei].unsqueeze(0), encoder_hidden)

    # Decoding
    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden

    for di in range(target_length):
        decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
        _, topi = decoder_output.topk(1)
        decoder_input = topi.squeeze().detach()

        loss += criterion(decoder_output, target_tensor[di].unsqueeze(0))

        if decoder_input.item() == EOS_token:
            break

    return loss / target_length

In [ ]:
def train(input_tensor, target_tensor, encoder, decoder, encoder_optim, decoder_optim, criterion):
    encoder_optim.zero_grad()
    decoder_optim.zero_grad()

    loss = forward_pass(input_tensor, target_tensor, encoder, decoder, criterion)
    loss.backward()

    encoder_optim.step()
    decoder_optim.step()

    return loss.item()

In [ ]:
# Taken from sequence2sequence.py
def eval_show_eg(encoder, decoder, n_eg=5):
    encoder.eval()
    decoder.eval()

    exact_matches = 0
    total_bleu_score = 0.0
    chencherry = SmoothingFunction()

    print(f'\n')
    print('='*70)
    print(f'Generating Validation Examples')
    print('='*70)

    with torch.no_grad():
        for i, (input_tensor, target_tensor) in enumerate(val_reversed_loader):
            input_tensor = input_tensor[0].to(device)
            target_tensor = target_tensor[0].to(device)

            encoder_hidden = encoder.initHidden()
            input_length = input_tensor.size(0)

            # Passing through encoder
            for ei in range(input_length):
                encoder_output, encoder_hidden = encoder(input_tensor[ei].unsqueeze(0), encoder_hidden)
            
            # Setting up the decoder
            decoder_input = torch.tensor([[SOS_token]], device=device)
            decoder_hidden = encoder_hidden
            predicted_idx = []

            # Range is extended to account for the larger sentences
            for di in range(100):
                decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
                _, topi = decoder_output.topk(1)
                
                idx_pred = topi.item()
                predicted_idx.append(idx_pred)
                
                decoder_input = topi.squeeze().detach()

                if idx_pred == EOS_token:
                    break

            input_string = ''.join([idx_to_chars[index.item()] for index in input_tensor if index.item() not in (SOS_token, EOS_token)])
            target_string = ''.join([idx_to_chars[index.item()] for index in target_tensor if index.item() not in (SOS_token, EOS_token)])
            predicted_string = ''.join([idx_to_chars[index] for index in predicted_idx if index not in (SOS_token, EOS_token)])

            # Metric 1: Tradition Sequence Accuracy
            if predicted_string == target_string:
                exact_matches += 1
            
            # Metric 2: BLUE Score
            ref_tokens = [list(target_string)]
            candidate_tokens = list(predicted_string)

            bleu = sentence_bleu(ref_tokens, candidate_tokens, smoothing_function=chencherry.method1)
            total_bleu_score += bleu

            if i < n_eg:
                match_status = "PASS" if predicted_string == target_string else "FAIL"
                print(f'Input: {input_string:<12} | Target: {target_string:<12} | Predicted: {predicted_string:<12} | Match: {match_status:<4} | BLEU: {bleu:.4f}')
        
        total_samples = len(val_reversed_loader)
        final_acc = exact_matches / total_samples
        avg_bleu = total_bleu_score / total_samples

        print('='*70)
        print(f'Final Metrics across Dataset:')
        print(f'\nTraditional Exact-Match Accuracy: {final_acc * 100:.2f}% ({exact_matches} / {total_samples} samples)')
        print(f'\nAverage Validation BLEU Score: {avg_bleu:.4f}')
        print('='*70)

## Attention Helper Functions

In [ ]:
def forward_pass_attn(input_tensor, target_tensor, encoder, decoder, criterion):
    encoder_hidden = encoder.initHidden()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)
    loss = 0

    # Encoding

    # Collecting outputs for attention
    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

    # Encoding loop
    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei].unsqueeze(0), encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    # Decoding
    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden

    for di in range(target_length):
        decoder_output, decoder_hidden, attn_weights = decoder(decoder_input, decoder_hidden, encoder_outputs)
        _, topi = decoder_output.topk(1)
        decoder_input = topi.squeeze().detach()

        loss += criterion(decoder_output, target_tensor[di].unsqueeze(0))

        if decoder_input.item() == EOS_token:
            break

    return loss / target_length

In [ ]:
def train_attn(input_tensor, target_tensor, encoder, decoder, encoder_optim, decoder_optim, criterion):
    encoder_optim.zero_grad()
    decoder_optim.zero_grad()

    loss = forward_pass_attn(input_tensor, target_tensor, encoder, decoder, criterion)
    loss.backward()

    encoder_optim.step()
    decoder_optim.step()

    return loss.item()

In [ ]:
# Taken from sequence2sequence.py
def eval_show_eg_attn(encoder, decoder, n_eg=5):
    encoder.eval()
    decoder.eval()

    exact_matches = 0
    total_bleu_score = 0.0
    chencherry = SmoothingFunction()

    print(f'\n')
    print('='*70)
    print(f'Generating Validation Examples')
    print('='*70)

    with torch.no_grad():
        for i, (input_tensor, target_tensor) in enumerate(val_reversed_loader):
            input_tensor = input_tensor[0].to(device)
            target_tensor = target_tensor[0].to(device)

            encoder_hidden = encoder.initHidden()
            input_length = input_tensor.size(0)
            encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

            # Passing through encoder
            for ei in range(input_length):
                encoder_output, encoder_hidden = encoder(input_tensor[ei].unsqueeze(0), encoder_hidden)
                encoder_outputs[ei] = encoder_output[0, 0]
            
            # Setting up the decoder
            decoder_input = torch.tensor([[SOS_token]], device=device)
            decoder_hidden = encoder_hidden
            predicted_idx = []

            # Range is extended to account for the larger sentences
            for di in range(100):
                decoder_output, decoder_hidden, attn_weights = decoder(decoder_input, decoder_hidden, encoder_outputs)
                _, topi = decoder_output.topk(1)
                
                idx_pred = topi.item()
                predicted_idx.append(idx_pred)
                
                decoder_input = topi.squeeze().detach()

                if idx_pred == EOS_token:
                    break

            input_string = ''.join([idx_to_chars[index.item()] for index in input_tensor if index.item() not in (SOS_token, EOS_token)])
            target_string = ''.join([idx_to_chars[index.item()] for index in target_tensor if index.item() not in (SOS_token, EOS_token)])
            predicted_string = ''.join([idx_to_chars[index] for index in predicted_idx if index not in (SOS_token, EOS_token)])

            # Metric 1: Tradition Sequence Accuracy
            if predicted_string == target_string:
                exact_matches += 1
            
            # Metric 2: BLUE Score
            ref_tokens = [list(target_string)]
            candidate_tokens = list(predicted_string)

            bleu = sentence_bleu(ref_tokens, candidate_tokens, smoothing_function=chencherry.method1)
            total_bleu_score += bleu

            if i < n_eg:
                match_status = "PASS" if predicted_string == target_string else "FAIL"
                print(f'Input: {input_string:<12} | Target: {target_string:<12} | Predicted: {predicted_string:<12} | Match: {match_status:<4} | BLEU: {bleu:.4f}')
        
        total_samples = len(val_reversed_loader)
        final_acc = exact_matches / total_samples
        avg_bleu = total_bleu_score / total_samples

        print('='*70)
        print(f'Final Metrics across Dataset:')
        print(f'\nTraditional Exact-Match Accuracy: {final_acc * 100:.2f}% ({exact_matches} / {total_samples} samples)')
        print(f'\nAverage Validation BLEU Score: {avg_bleu:.4f}')
        print('='*70)

## Running the Baseline

In [ ]:
encoder_3a = Encoder(inp_size=input_size, hidden_size=hidden_size).to(device)
decoder_3a = Decoder(hidden_size=hidden_size, out_size=output_size).to(device)

learning_rate = 0.001
encoder_optimiser_3a = optim.Adam(encoder_3a.parameters(), lr=learning_rate)
decoder_optimiser_3a = optim.Adam(decoder_3a.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

# Initialising loss variables
train_losses_3a = []
val_losses_3a = []

print("="*80)
print("Starting Baseline Training")
print("="*80)

# Training and Validation Loop
for epoch in range(no_of_epochs):
    # TRAINING
    encoder_3a.train()
    decoder_3a.train()
    total_train_loss = 0

    for input_tensor, target_tensor in train_reversed_loader:
        input_tensor = input_tensor[0].to(device)
        target_tensor = target_tensor[0].to(device)

        loss = train(input_tensor, target_tensor, encoder_3a, decoder_3a, encoder_optimiser_3a, decoder_optimiser_3a, criterion)
        total_train_loss += loss
    
    avg_train_loss = total_train_loss / len(train_reversed_loader)
    train_losses_3a.append(avg_train_loss)

    # VALIDATION
    encoder_3a.eval()
    decoder_3a.eval()
    total_val_loss = 0

    with torch.no_grad():
        for input_tensor, target_tensor in val_reversed_loader:
            input_tensor = input_tensor[0].to(device)
            target_tensor = target_tensor[0].to(device)

            encoder_hidden = encoder_3a.initHidden()
            input_length = input_tensor.size(0)
            target_length = target_tensor.size(0)
            loss = forward_pass(input_tensor, target_tensor, encoder_3a, decoder_3a, criterion)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_reversed_loader)
    val_losses_3a.append(avg_val_loss)

    if epoch % 5 == 0:
        print(f'Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

eval_show_eg(encoder_3a, decoder_3a, n_eg=5)
plot_loss(train_losses=train_losses_3a, val_losses=val_losses_3a, title="Part 3a: GRU Baseline Encoder-Decoder Loss Curves for French to English", path=f'{out_path}/data/baseline_loss_curves_reversed')

## Running with Attention

In [ ]:
encoder_3b = Encoder(inp_size=input_size, hidden_size=hidden_size).to(device)
decoder_3b = AttnDecoder(hidden_size=hidden_size, out_size=output_size).to(device)

learning_rate = 0.001
encoder_optimiser_3b = optim.Adam(encoder_3b.parameters(), lr=learning_rate)
decoder_optimiser_3b = optim.Adam(decoder_3b.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

# Initialising loss variables
train_losses_3b = []
val_losses_3b = []

print("="*80)
print("Starting Attention Training")
print("="*80)

# Training and Validation Loop
for epoch in range(no_of_epochs):
    # TRAINING
    encoder_3b.train()
    decoder_3b.train()
    total_train_loss = 0

    for input_tensor, target_tensor in train_reversed_loader:
        input_tensor = input_tensor[0].to(device)
        target_tensor = target_tensor[0].to(device)

        loss = train_attn(input_tensor, target_tensor, encoder_3b, decoder_3b, encoder_optimiser_3b, decoder_optimiser_3b, criterion)
        total_train_loss += loss
    
    avg_train_loss = total_train_loss / len(train_reversed_loader)
    train_losses_3b.append(avg_train_loss)

    # VALIDATION
    encoder_3b.eval()
    decoder_3b.eval()
    total_val_loss = 0

    with torch.no_grad():
        for input_tensor, target_tensor in val_reversed_loader:
            input_tensor = input_tensor[0].to(device)
            target_tensor = target_tensor[0].to(device)

            encoder_hidden = encoder_3b.initHidden()
            input_length = input_tensor.size(0)
            target_length = target_tensor.size(0)
            loss = forward_pass_attn(input_tensor, target_tensor, encoder_3b, decoder_3b, criterion)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_reversed_loader)
    val_losses_3b.append(avg_val_loss)

    if epoch % 5 == 0:
        print(f'Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}')

eval_show_eg_attn(encoder_3b, decoder_3b, n_eg=5)
plot_loss(train_losses=train_losses_3b, val_losses=val_losses_3b, title="Part 3b: GRU Baseline Encoder-Decoder (with Attention) Loss Curves for French to English", path=f'{out_path}/data/attention_loss_curves')